# 📝 과제 01: 온라인쇼핑 조문 검색기를 MNR 손실로 파인튜닝하기

## 만들 결과

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;">교안 01에서 노무 조문으로 진행한 쌍 학습을 온라인쇼핑(전자상거래법) 조문 검색에 같은 순서로 적용하고, 학습 전·후 검색 품질을 비교합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">만들 것</strong></p>
<ul>
<li>세 데이터 리스트 <code style="font-size:0.95em;color:#345473;">commerce_corpus</code>, <code style="font-size:0.95em;color:#345473;">commerce_train</code>, <code style="font-size:0.95em;color:#345473;">commerce_eval</code></li>
<li>베이스 모델의 정답 순위 <code style="font-size:0.95em;color:#345473;">base_ranks</code>와 점수 <code style="font-size:0.95em;color:#345473;">base_scores</code></li>
<li>학습 데이터셋 <code style="font-size:0.95em;color:#345473;">pair_ds</code>, <code style="font-size:0.95em;color:#345473;">BatchSamplers.NO_DUPLICATES</code>로 학습한 쌍 학습 모델 <code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code></li>
<li>학습 전·후 점수 테이블 <code style="font-size:0.95em;color:#345473;">score_table</code>, 처음 보는 조문 질의를 나눈 테이블 <code style="font-size:0.95em;color:#345473;">split_table</code></li>
<li>저장한 모델 폴더 <code style="font-size:0.95em;color:#345473;">output/commerce_tuned</code>(과제 02가 불러옵니다)와 새 질의의 상위 5개 조문 테이블 <code style="font-size:0.95em;color:#345473;">search_table</code></li>
<li>일반 83질의의 모델별 정답 순위 <code style="font-size:0.95em;color:#345473;">general_model_ranks</code>와 전체·분야별 학습 전·후 테이블 <code style="font-size:0.95em;color:#345473;">general_score_table</code></li>
</ul></div>

## 준비: 데이터와 제공 코드

- **자료:** `data/과제/commerce_corpus.jsonl`(조문 39개: `doc_id`, `law`, `topic`, `text`), `commerce_train.jsonl`(학습 쌍 82개: `anchor`, `positive`, `doc_id`), `commerce_eval.jsonl`(검증용 정답 쌍 46개: `anchor`, `positive`, `doc_id`). 문서 한 건은 조문 항(예: `전자상거래법 제17조 제1항`)입니다.
- **일반 평가 자료:** 마지막 8번에서 `data/본실습/general_corpus.jsonl`(433문서), `general_eval.jsonl`(83질의)을 읽습니다.
- **제공 코드:** 준비 셀(`data_dir`, `output_dir`, `device`, `MODEL_ID`, `QUERY_PREFIX`, `PASSAGE_PREFIX`)과 교안 01과 같은 함수 `load_jsonl`, `encode_texts`, `gold_ranks`(정답 순위 배열), `retrieval_scores`(점수 딕셔너리)
- **작성:** `# TODO` 아래 `...`를 채우고 문항 끝의 `[자가채점]` 셀을 실행합니다. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.

In [ ]:
from pathlib import Path

import pandas as pd
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 조문이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 이 단원의 임베딩 모델과 e5 계열 프리픽스(질의는 "query: ", 문서는 "passage: ")
MODEL_ID = "intfloat/multilingual-e5-small"
QUERY_PREFIX = "query: "
PASSAGE_PREFIX = "passage: "

In [ ]:
import json


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def encode_texts(model, texts, prefix):
    """텍스트 목록에 프리픽스를 붙여 정규화된 임베딩(numpy 배열)으로 만듭니다."""
    return model.encode(
        [prefix + text for text in texts],   # 프리픽스를 붙인 문장 목록
        normalize_embeddings=True,           # 길이 1로 정규화해 벡터끼리의 내적이 곧 코사인 유사도가 되게 합니다
        show_progress_bar=False,             # 진행 막대를 끕니다(평가 때마다 여러 번 불러 출력이 길어지지 않게)
    )

In [ ]:
import numpy as np


def gold_ranks(model, corpus, evalset, query_prefix=QUERY_PREFIX, passage_prefix=PASSAGE_PREFIX):
    """평가 질의마다 정답 문서가 유사도 순으로 몇 위인지(1위부터) 담은 배열을 돌려줍니다."""
    doc_ids = [doc["doc_id"] for doc in corpus]
    doc_emb = encode_texts(model, [doc["text"] for doc in corpus], passage_prefix)
    # 학습·평가 파일은 같은 세 열을 쓰며, 검색 평가는 질의와 정답 ID를 사용합니다.
    query_emb = encode_texts(model, [row["anchor"] for row in evalset], query_prefix)

    # 정규화한 벡터끼리의 내적이 코사인 유사도입니다. sims는 (질의 수, 문서 수) 행렬입니다
    sims = query_emb @ doc_emb.T

    # 질의마다 유사도가 높은 순으로 문서 id를 늘어놓고, 정답 문서가 몇 번째인지 셉니다
    ranks = []
    for i, row in enumerate(evalset):
        ranked_ids = [doc_ids[j] for j in np.argsort(-sims[i])]
        ranks.append(ranked_ids.index(row["doc_id"]) + 1)
    return np.array(ranks)


def retrieval_scores(ranks):
    """정답 순위 배열로 Recall@1, Recall@5, MRR을 계산합니다."""
    return {
        "Recall@1": round(float(np.mean(ranks <= 1)), 3),  # 정답 문서가 1위인 질의의 비율
        "Recall@5": round(float(np.mean(ranks <= 5)), 3),  # 정답 문서가 5위 안에 든 질의의 비율
        "MRR": round(float(np.mean(1 / ranks)), 3),        # 평균(1 / 정답 순위)
    }

## 1. 조문과 질의 데이터 읽기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 학습과 평가에 쓸 조문, 학습 쌍, 평가 질의를 준비합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) <code style="font-size:0.95em;color:#345473;">data_dir / &quot;과제&quot;</code> 아래 세 파일을 <code style="font-size:0.95em;color:#345473;">load_jsonl</code>로 읽어 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">commerce_corpus</code></strong>, <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">commerce_train</code></strong>, <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">commerce_eval</code></strong> 에 담습니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01 1-2 기존 질문과 정답 조문 읽기 셀에서 폴더를 "과제"로, 파일 이름을 commerce_ 로 바꿉니다.
```

</details>

In [ ]:
# TODO (1): data_dir / "과제" 아래 세 파일을 load_jsonl 로 읽으세요
commerce_corpus = ...
commerce_train = ...
commerce_eval = ...
print("조문:", len(commerce_corpus), "| 학습 쌍:", len(commerce_train), "| 평가 질의:", len(commerce_eval))

In [ ]:
# [자가채점] 세 파일의 행 수와 키를 확인합니다
assert isinstance(commerce_corpus, list) and len(commerce_corpus) == 39, "commerce_corpus는 commerce_corpus.jsonl 을 load_jsonl 로 읽은 39행 리스트입니다. 경로가 data_dir / 과제 아래인지 확인하세요."
assert isinstance(commerce_train, list) and len(commerce_train) == 82, "commerce_train은 commerce_train.jsonl 을 읽은 82행 리스트입니다. 파일 이름을 확인하세요."
assert isinstance(commerce_eval, list) and len(commerce_eval) == 46, "commerce_eval은 commerce_eval.jsonl 을 읽은 46행 리스트입니다. 파일 이름을 확인하세요."
assert list(commerce_train[0]) == ["anchor", "positive", "doc_id"], f"commerce_train 행의 키가 {list(commerce_train[0])}입니다. 학습 파일의 열 이름과 순서를 확인하세요."
assert list(commerce_eval[0]) == ["anchor", "positive", "doc_id"], f"commerce_eval 행의 키가 {list(commerce_eval[0])}입니다. 평가 파일을 읽었는지 확인하세요."
print("✅ 통과!")

## 2. 베이스 모델의 출발점

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 학습 효과를 비교할 출발점을 만들기 위해, 학습 전 모델이 평가 질의마다 정답 조문을 몇 위에 두는지 계산합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) <code style="font-size:0.95em;color:#345473;">MODEL_ID</code>로 만들어 <code style="font-size:0.95em;color:#345473;">device</code>에 올린 <code style="font-size:0.95em;color:#345473;">SentenceTransformer</code>를 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">base_model</code></strong> 에 담습니다.</li>
<li>(2) <code style="font-size:0.95em;color:#345473;">gold_ranks</code>에 <code style="font-size:0.95em;color:#345473;">base_model</code>, <code style="font-size:0.95em;color:#345473;">commerce_corpus</code>, <code style="font-size:0.95em;color:#345473;">commerce_eval</code>을 넘겨(프리픽스 인자는 기본값) 정답 순위 배열 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">base_ranks</code></strong> 를 만듭니다.</li>
<li>(3) <code style="font-size:0.95em;color:#345473;">base_ranks</code>를 <code style="font-size:0.95em;color:#345473;">retrieval_scores</code>에 넘겨 돌려받은 딕셔너리를 테이블로 감싸지 않고 그대로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">base_scores</code></strong> 에 담습니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01 2-1 베이스 모델 셀과 2-2 점수 셀과 같은 방법입니다.
교안과 달리 점수를 DataFrame 으로 감싸지 않고 retrieval_scores 의 반환값을 그대로 담습니다.
```

</details>

In [ ]:
from sentence_transformers import SentenceTransformer

# 모델을 처음 내려받을 때 HF_TOKEN 경고가 나올 수 있습니다. 로그인 없이 공개 모델을 받는다는 안내라 그대로 진행합니다

# TODO (1): MODEL_ID 로 베이스 모델을 만들어 device 에 올리세요
base_model = ...

# TODO (2): 평가 질의 46개마다 정답 조문의 순위(1위부터)를 구하세요
base_ranks = ...

# TODO (3): Recall@1, Recall@5, MRR 딕셔너리를 계산하세요
base_scores = ...
display(base_scores)

In [ ]:
# [자가채점] 베이스 모델, 순위 배열의 모양과 범위, 점수 딕셔너리를 확인합니다
assert isinstance(base_model, SentenceTransformer), "base_model은 MODEL_ID 로 만든 SentenceTransformer 입니다. `...`를 채웠는지 확인하세요."
assert isinstance(base_ranks, np.ndarray) and base_ranks.shape == (46,), f"base_ranks의 모양이 {np.shape(base_ranks)}입니다. 평가 질의 46개마다 순위 하나인 gold_ranks 의 반환값을 담으세요."
assert base_ranks.min() >= 1 and base_ranks.max() <= 39, "순위는 1부터 조문 수 39까지입니다. corpus 자리에 commerce_corpus 를 넘겼는지 확인하세요."
assert isinstance(base_scores, dict) and list(base_scores) == ["Recall@1", "Recall@5", "MRR"], "base_scores는 retrieval_scores 가 돌려준 딕셔너리입니다. DataFrame 으로 감싸지 말고 그대로 담으세요."
for value in base_scores.values():
    assert 0 <= value <= 1, f"점수 {value}가 0~1 밖입니다. retrieval_scores 에 base_ranks 를 넘겼는지 확인하세요."
assert np.array_equal(base_ranks, gold_ranks(base_model, commerce_corpus, commerce_eval)), "base_ranks가 베이스 모델의 실제 검색 순위와 다릅니다. 모델과 평가 자료를 확인하세요."
print("✅ 통과!")

## 3. 학습 데이터셋 pair_ds

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 학습기에 넣을 학습 데이터를 만들기 위해, 학습 쌍을 프리픽스를 붙인 <code style="font-size:0.95em;color:#345473;">Dataset</code>으로 바꿉니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) 열이 <code style="font-size:0.95em;color:#345473;">&quot;anchor&quot;</code>(<code style="font-size:0.95em;color:#345473;">QUERY_PREFIX</code> + 질의), <code style="font-size:0.95em;color:#345473;">&quot;positive&quot;</code>(<code style="font-size:0.95em;color:#345473;">PASSAGE_PREFIX</code> + 정답 문서) 두 개(이 순서, <code style="font-size:0.95em;color:#345473;">doc_id</code> 열 없음)이고 행이 <code style="font-size:0.95em;color:#345473;">commerce_train</code> 순서 그대로인 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">pair_ds</code></strong> 를 만듭니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01 1-3 학습용 쌍 데이터셋 만들기 셀과 같은 방법입니다.
```

</details>

In [ ]:
from datasets import Dataset

# TODO (1): anchor 에는 질의 프리픽스, positive 에는 문서 프리픽스를 붙인 두 열짜리 데이터셋을 만드세요
pair_ds = ...
display(pair_ds)

In [ ]:
# [자가채점] 열 이름과 행 수, 프리픽스, 행 순서를 확인합니다
assert isinstance(pair_ds, Dataset), "pair_ds는 datasets 의 Dataset 입니다. `...`를 채웠는지 확인하세요."
assert pair_ds.column_names == ["anchor", "positive"], f"pair_ds의 열이 {pair_ds.column_names}입니다. anchor, positive 두 열만 이 순서로 둡니다(doc_id 는 넣지 않습니다)."
assert len(pair_ds) == 82, f"pair_ds가 {len(pair_ds)}행입니다. commerce_train 82행을 모두 넣으세요."
for anchor, positive in zip(pair_ds["anchor"], pair_ds["positive"]):
    assert anchor.startswith(QUERY_PREFIX) and not anchor.startswith(QUERY_PREFIX + QUERY_PREFIX), f"anchor는 query: 로 한 번만 시작합니다: {anchor[:30]}. QUERY_PREFIX 를 한 번만 붙였는지 확인하세요."
    assert positive.startswith(PASSAGE_PREFIX) and not positive.startswith(PASSAGE_PREFIX + PASSAGE_PREFIX), f"positive는 passage: 로 한 번만 시작합니다: {positive[:30]}. PASSAGE_PREFIX 를 한 번만 붙였는지 확인하세요."
assert pair_ds[0]["anchor"] == QUERY_PREFIX + commerce_train[0]["anchor"], "첫 행의 anchor가 commerce_train 첫 행의 질의와 다릅니다. 행 순서를 바꾸지 마세요."
assert pair_ds[0]["positive"] == PASSAGE_PREFIX + commerce_train[0]["positive"], "첫 행의 positive가 commerce_train 첫 행의 정답 문서(positive)와 다릅니다. doc_id 가 아니라 positive 의 문장을 넣으세요."
print("✅ 통과!")

## 4. NO_DUPLICATES 배치로 쌍 학습

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 온라인쇼핑 조문 표현에 맞춘 검색 모델을 얻기 위해, 베이스 모델과 같은 모델에서 새로 시작해 <code style="font-size:0.95em;color:#345473;">BatchSamplers.NO_DUPLICATES</code> 배치와 MNR 손실로 학습합니다.</p></div>

최대 5에폭 동안 매 에폭 검증·저장하고, 검증 `Recall@5`가 2회 연속 개선되지 않으면 일찍 종료합니다. 학습이 끝나면 **검증 Recall@5가 가장 높았던 체크포인트**를 복원합니다. 검증용 `commerce_eval`은 모델 선택에 사용하므로 독립적인 최종 테스트 점수로 해석하지 않으며, 일반 83질의는 모델 선택에 쓰지 않습니다.

검증 쌍의 MNR 손실 `eval_loss`도 로그에 기록합니다. 이 손실은 검증 배치 구성의 영향을 받으므로, 손실 최저가 아니라 전체 조문 검색의 Recall@5로 모델을 선택합니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) <code style="font-size:0.95em;color:#345473;">MODEL_ID</code>로 새로 만들어 <code style="font-size:0.95em;color:#345473;">device</code>에 올린 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code></strong>(<code style="font-size:0.95em;color:#345473;">base_model</code>과 다른 객체)을 만듭니다.</li>
<li>(2) <code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code>을 넘긴 <code style="font-size:0.95em;color:#345473;">MultipleNegativesRankingLoss</code>로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">loss</code></strong> 를 만듭니다.</li>
<li>(3) 아래 값의 <code style="font-size:0.95em;color:#345473;">SentenceTransformerTrainingArguments</code>를 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">args</code></strong> 에 담습니다.</li>
</ul></div>

| 인자 | 값 |
|---|---|
| `output_dir` | `str(output_dir / "commerce_tuned_run")` |
| `num_train_epochs` | 5(최대) |
| `per_device_train_batch_size` | 16 |
| `per_device_eval_batch_size` | 32 |
| `learning_rate` | 5e-5 |
| `warmup_steps` | 0.1 |
| `batch_sampler` | `BatchSamplers.NO_DUPLICATES` |
| `eval_strategy` | `"epoch"` |
| `save_strategy` | `"epoch"` |
| `load_best_model_at_end` | `True` |
| `metric_for_best_model` | `"eval_commerce_cosine_recall@5"` |
| `greater_is_better` | `True` |
| `save_total_limit` | 2 |
| `logging_steps` | 5 |
| `report_to` | `"none"` |
| `seed` | 42 |
| `use_cpu` | `device.type == "cpu"` |

- (4) `model=commerce_tuned_model`, `args=args`, `train_dataset=pair_ds`, `eval_dataset=eval_ds`, `loss=loss`, `evaluator=retrieval_evaluator`, `callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]`를 넘긴 `SentenceTransformerTrainer`를 **`trainer`** 에 담습니다. `eval_ds`와 `retrieval_evaluator`는 제공 코드입니다.
- (5) `trainer`로 학습한 반환값을 **`train_output`** 에 담습니다. 제공 코드의 `epoch_history`는 에폭별 `eval_loss`·검증 Recall@5를, `best_checkpoint_table`은 종료 에폭·최적 에폭·체크포인트·최적 검증 Recall@5를 표시합니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 4-1~4-3의 모델·손실·검증·학습 설정과 같은 순서입니다.
eval_dataset은 검증 손실, retrieval_evaluator는 전체 조문 검색 점수를 계산합니다. 콜백은 검증 Recall@5의 개선을 기다리는 횟수를 정합니다.
```

</details>

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss
from sentence_transformers.sentence_transformer.training_args import BatchSamplers
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.util import cos_sim
from transformers import EarlyStoppingCallback

# [제공] 검증 쌍은 eval_loss를 기록하는 데만 쓰며 가중치 갱신에는 쓰지 않습니다.
eval_ds = Dataset.from_dict({
    "anchor": [QUERY_PREFIX + row["anchor"] for row in commerce_eval],
    "positive": [PASSAGE_PREFIX + row["positive"] for row in commerce_eval],
})
# 전체 쇼핑 조문에서 검증 질의의 정답 순위를 구해 Recall@5로 모델을 선택합니다.
retrieval_evaluator = InformationRetrievalEvaluator(
    # 질의 ID는 정답 매핑과 연결할 문자열 키이고, 값은 프리픽스가 붙은 질문입니다.
    queries={str(i): QUERY_PREFIX + row["anchor"] for i, row in enumerate(commerce_eval)},
    # 문서 ID -> 프리픽스가 붙은 검색 문서입니다.
    corpus={doc["doc_id"]: PASSAGE_PREFIX + doc["text"] for doc in commerce_corpus},
    # 질의 ID -> 정답 조문 ID 집합입니다.
    relevant_docs={str(i): {row["doc_id"]} for i, row in enumerate(commerce_eval)},
    name="commerce",  # 지표 이름에 포함되어 eval_commerce_cosine_recall@5를 만듭니다.
    score_functions={"cosine": cos_sim},  # 문서 점수는 코사인 유사도로 비교합니다.
    precision_recall_at_k=[1, 5],  # Recall@1과 Recall@5를 계산합니다.
    batch_size=64,  # 평가 문장을 64개씩 인코딩합니다.
    write_csv=False,
)

# TODO (1): MODEL_ID 에서 새 쌍 학습 모델을 만드세요
commerce_tuned_model = ...

# TODO (2): 새 모델의 MNR 손실을 만드세요
loss = ...

# TODO (3): 검증·저장·최적 모델 복원에 필요한 빈칸을 요구사항 테이블대로 채우세요
args = SentenceTransformerTrainingArguments(
    output_dir=str(output_dir / "commerce_tuned_run"),
    num_train_epochs=...,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=...,
    learning_rate=5e-5,
    warmup_steps=0.1,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy=...,
    save_strategy=...,
    load_best_model_at_end=...,
    # evaluator가 만든 검증 지표 이름입니다. name·점수 방식·Recall@K 설정에서 정해집니다.
    metric_for_best_model=...,
    greater_is_better=...,                     # Recall@5는 값이 클수록 좋은 모델입니다.
    save_total_limit=...,
    logging_steps=5,
    report_to="none",
    seed=42,
    use_cpu=(device.type == "cpu"),  # GPU(cuda)이면 False, CPU이면 True
)

# TODO (4): 검증 데이터, 검색 평가기, patience 2 콜백을 채워 학습기를 만드세요
trainer = SentenceTransformerTrainer(
    model=commerce_tuned_model,
    args=args,
    train_dataset=pair_ds,
    eval_dataset=...,
    loss=loss,
    evaluator=...,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=...)],
)

In [ ]:
# 학습을 시작하면 [transformers] 토크나이저 PAD/BOS/EOS 안내가 한 번 나옵니다. 모델 설정을 토크나이저에 맞췄다는 안내라 오류가 아닙니다.
# TODO (5): 학습하세요. 반환값의 global_step 은 실제 갱신 횟수입니다.
train_output = ...
display(train_output)

# [제공] 에폭별 검증 손실과 검색 점수, 복원한 최적 체크포인트를 확인합니다.
eval_logs = [log for log in trainer.state.log_history if "eval_commerce_cosine_recall@5" in log]
epoch_history = pd.DataFrame([
    {"에폭": log["epoch"], "검증 MNR 손실": log["eval_loss"],
     "검증 Recall@5": log["eval_commerce_cosine_recall@5"]}
    for log in eval_logs
])
display(epoch_history.round(3))
best_step = int(Path(trainer.state.best_model_checkpoint).name.rsplit("-", 1)[1])
best_epoch = next(log["epoch"] for log in eval_logs if log["step"] == best_step)
best_checkpoint_table = pd.DataFrame([{
    "선택 기준": "검증 Recall@5 최대",
    "최적 Recall@5": trainer.state.best_metric,
    "복원 체크포인트": trainer.state.best_model_checkpoint,
    "종료 에폭": trainer.state.epoch,
    "최적 에폭": best_epoch,
}])
display(best_checkpoint_table.round(3))

In [ ]:
# [자가채점] 설정과 실제 검증 기록을 확인합니다. 조기 종료와 점수의 상승·하락을 허용합니다.
assert isinstance(commerce_tuned_model, SentenceTransformer) and commerce_tuned_model is not base_model, "commerce_tuned_model은 MODEL_ID로 새로 만든 모델입니다. 베이스 모델을 학습하지 마세요."
assert isinstance(loss, MultipleNegativesRankingLoss) and loss.model is commerce_tuned_model, "loss에 현재 학습할 모델을 넘겼는지 확인하세요."
assert isinstance(args, SentenceTransformerTrainingArguments), "args는 SentenceTransformerTrainingArguments입니다."
assert args.batch_sampler == BatchSamplers.NO_DUPLICATES, "batch_sampler=BatchSamplers.NO_DUPLICATES를 넣으세요."
assert args.num_train_epochs == 5 and args.per_device_train_batch_size == 16 and args.per_device_eval_batch_size == 32, "최대 5에폭, 학습 배치 16, 검증 배치 32를 넣으세요."
assert abs(args.learning_rate - 5e-5) < 1e-12 and args.warmup_steps == 0.1, "learning_rate=5e-5, warmup_steps=0.1을 넣으세요."
assert args.eval_strategy == "epoch" and args.save_strategy == "epoch", "매 에폭 검증·저장하도록 eval_strategy와 save_strategy를 모두 epoch으로 둡니다."
assert args.load_best_model_at_end and args.metric_for_best_model == "eval_commerce_cosine_recall@5" and args.greater_is_better, "검증 Recall@5가 가장 높은 체크포인트를 학습 종료 후 복원하도록 설정하세요."
assert args.save_total_limit == 2 and args.logging_steps == 5 and args.seed == 42, "save_total_limit=2, logging_steps=5, seed=42를 넣으세요."
assert len(args.report_to) == 0 and args.use_cpu == (device.type == "cpu"), "report_to=none, use_cpu=(device.type == cpu)의 요구사항을 확인하세요."
assert args.output_dir.endswith("commerce_tuned_run"), "output_dir에 요구사항의 실행 폴더 경로를 넣으세요."
assert isinstance(trainer, SentenceTransformerTrainer) and trainer.model is commerce_tuned_model and trainer.args is args, "trainer에 현재 모델과 args를 넘기세요."
assert trainer.train_dataset.column_names == ['anchor', 'positive'] and len(trainer.train_dataset) == len(pair_ds), "train_dataset에 pair_ds를 넘기세요. 검증 쌍은 학습 데이터가 아닙니다."
assert trainer.eval_dataset.column_names == ["anchor", "positive"] and len(trainer.eval_dataset) == 46, "eval_dataset에 제공한 두 열짜리 eval_ds 46행을 넘기세요."
assert trainer.evaluator is retrieval_evaluator, "전체 commerce_corpus를 검색하는 제공 retrieval_evaluator를 trainer에 넘기세요."
assert any(isinstance(callback, EarlyStoppingCallback) and callback.early_stopping_patience == 2 for callback in trainer.callback_handler.callbacks), "callbacks에 patience 2인 EarlyStoppingCallback을 넣으세요."
assert 0 < train_output.global_step <= 30 and 0 < trainer.state.epoch <= 5, "학습이 실행됐는지 확인하세요. 최대 5에폭 안에서 조기 종료할 수 있습니다."
assert 1 <= len(epoch_history) <= 5 and epoch_history["검증 MNR 손실"].notna().all(), "매 에폭 검증 손실이 기록되도록 eval_dataset과 eval_strategy를 확인하세요."
assert trainer.state.best_model_checkpoint and Path(trainer.state.best_model_checkpoint).is_dir(), "최적 체크포인트가 저장되지 않았습니다. epoch 저장·검증과 load_best_model_at_end를 확인하세요."
assert abs(trainer.state.best_metric - epoch_history["검증 Recall@5"].max()) < 1e-12, "최적 검증 점수는 기록된 Recall@5의 최댓값이어야 합니다. eval_loss로 선택하지 마세요."
print("✅ 통과!")

## 5. 학습 전·후 점수 테이블

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 학습 효과를 수치로 비교하기 위해, 쌍 학습 모델의 정답 순위를 구해 베이스 모델과 한 테이블에 둡니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) 2번과 같은 방법으로 <code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code>의 정답 순위 배열 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">tuned_ranks</code></strong> 를 만듭니다.</li>
<li>(2) 행 이름(인덱스)이 <code style="font-size:0.95em;color:#345473;">&quot;베이스 모델&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;쌍 학습 모델&quot;</code>, 열이 <code style="font-size:0.95em;color:#345473;">&quot;Recall@1&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;Recall@5&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;MRR&quot;</code>(이 순서)인 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">score_table</code></strong> 을 만듭니다. 베이스 모델 행은 <code style="font-size:0.95em;color:#345473;">base_scores</code>, 쌍 학습 모델 행은 <code style="font-size:0.95em;color:#345473;">tuned_ranks</code>의 <code style="font-size:0.95em;color:#345473;">retrieval_scores</code> 값입니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01 5-1 같은 평가셋의 점수 셀과 같은 방법입니다. 점수 딕셔너리 두 개를 행으로 쌓으면 딕셔너리의 키가 열 이름이 됩니다.
```

</details>

In [ ]:
# TODO (1): 2번과 같은 방법으로 쌍 학습 모델의 정답 순위를 구하세요
tuned_ranks = ...

# TODO (2): 행 베이스 모델, 쌍 학습 모델 / 열 Recall@1, Recall@5, MRR 인 테이블을 만드세요
score_table = ...
display(score_table)

In [ ]:
# [자가채점] 순위 배열과 테이블의 일관성을 확인합니다. 성능 향상을 강제하지 않습니다.
assert isinstance(tuned_ranks, np.ndarray) and tuned_ranks.shape == (46,), f"tuned_ranks의 모양이 {np.shape(tuned_ranks)}입니다. gold_ranks 의 반환값을 담으세요."
assert tuned_ranks.min() >= 1 and tuned_ranks.max() <= 39, "순위는 1부터 조문 수 39까지입니다. corpus 자리에 commerce_corpus 를 넘겼는지 확인하세요."
assert isinstance(score_table, pd.DataFrame), "score_table은 pd.DataFrame 입니다. 점수 딕셔너리 두 개를 pd.DataFrame 에 넘겼는지 확인하세요."
assert list(score_table.index) == ["베이스 모델", "쌍 학습 모델"], f"score_table의 행 이름이 {list(score_table.index)}입니다. index 에 베이스 모델, 쌍 학습 모델을 이 순서로 넣으세요."
assert list(score_table.columns) == ["Recall@1", "Recall@5", "MRR"], f"score_table의 열이 {list(score_table.columns)}입니다. Recall@1, Recall@5, MRR 순서인지 확인하세요."
assert score_table.loc["베이스 모델"].to_dict() == base_scores, "베이스 모델 행이 2번의 base_scores 와 다릅니다. 첫 행에 base_scores 를 넣었는지 확인하세요."
assert np.array_equal(tuned_ranks, gold_ranks(commerce_tuned_model, commerce_corpus, commerce_eval)), "tuned_ranks가 최적 체크포인트를 복원한 쌍 학습 모델의 실제 검색 순위와 다릅니다."
assert score_table.loc["쌍 학습 모델"].to_dict() == retrieval_scores(tuned_ranks), "쌍 학습 모델 행은 tuned_ranks의 검색 지표를 담아야 합니다."
print("✅ 통과!")

## 6. 처음 보는 조문 질의 나누기

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 학습 효과가 학습한 조문에만 나타나는지 보기 위해, 정답 조문이 학습 쌍에 한 번도 나오지 않은 질의(처음 보는 조문 질의)를 따로 계산합니다.</p></div>

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) <code style="font-size:0.95em;color:#345473;">commerce_train</code>의 <code style="font-size:0.95em;color:#345473;">doc_id</code>를 모은 집합(set) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">train_doc_ids</code></strong> 를 만듭니다.</li>
<li>(2) 평가 질의마다 <code style="font-size:0.95em;color:#345473;">doc_id</code>가 <code style="font-size:0.95em;color:#345473;">train_doc_ids</code>에 없으면 <code style="font-size:0.95em;color:#345473;">True</code>인 numpy 불리언 배열 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">is_unseen</code></strong> 을 <code style="font-size:0.95em;color:#345473;">commerce_eval</code> 순서(길이 46)로 만듭니다.</li>
<li>(3) 셀에 적어 둔 <code style="font-size:0.95em;color:#345473;">query_groups</code>, <code style="font-size:0.95em;color:#345473;">model_ranks</code> 순서로 묶음마다 모델별 한 행씩, 열이 <code style="font-size:0.95em;color:#345473;">&quot;질의 묶음&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;모델&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;질의 수&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;Recall@1&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;Recall@5&quot;</code>, <code style="font-size:0.95em;color:#345473;">&quot;MRR&quot;</code>(이 순서)인 네 행 테이블 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">split_table</code></strong> 을 만듭니다. 점수는 그 묶음의 순위만 골라 <code style="font-size:0.95em;color:#345473;">retrieval_scores</code>로 계산합니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 02 6-1~6-2 질의 묶음별 비교 셀과 같은 방법입니다.
불리언 배열로 순위 배열을 인덱싱하면 True 인 질의의 순위만 남고, ~ 를 붙이면 반대 묶음이 남습니다.
```

</details>

In [ ]:
# TODO (1): 학습 쌍의 doc_id 집합을 만드세요
train_doc_ids = ...

# TODO (2): 평가 질의마다 정답 조문이 train_doc_ids 에 없으면 True 인 배열을 만드세요
is_unseen = ...

In [ ]:
# 질의 묶음과 모델(그대로 씁니다). 불리언 배열로 순위를 고르면 그 묶음 질의의 순위만 남습니다(~ 는 반대 묶음)
query_groups = {"학습에 나온 조문 질의": ~is_unseen, "처음 보는 조문 질의": is_unseen}
model_ranks = {"베이스 모델": base_ranks, "쌍 학습 모델": tuned_ranks}

# TODO (3): 묶음과 모델마다 점수를 계산해 행 딕셔너리를 split_rows 에 더하세요
split_rows = []
for group, mask in query_groups.items():
    for name, ranks in model_ranks.items():
        ...
split_table = pd.DataFrame(split_rows)
display(split_table)

In [ ]:
# [자가채점] 처음 보는 조문 질의 표시와 묶음별 테이블을 확인합니다
assert isinstance(train_doc_ids, set) and len(train_doc_ids) == 27 and "전자상거래법 제17조 제2항" in train_doc_ids, f"train_doc_ids가 원소 {len(train_doc_ids)}개입니다. commerce_train 의 doc_id 27개를 모은 집합(set)인지 확인하세요."
assert isinstance(is_unseen, np.ndarray) and is_unseen.dtype == bool and is_unseen.shape == (46,), "is_unseen은 평가 질의 46개마다 True/False 인 numpy 배열입니다. 리스트를 np.array 로 바꿨는지 확인하세요."
assert int(is_unseen.sum()) == 9, f"처음 보는 조문 질의가 {int(is_unseen.sum())}개입니다(기대값 9개). doc_id가 train_doc_ids 에 없을 때 True 인지 확인하세요."
assert list(np.flatnonzero(is_unseen)) == [2, 8, 10, 11, 21, 22, 25, 26, 27], "is_unseen의 True 위치가 다릅니다. commerce_eval 순서 그대로 만들었는지 확인하세요."
assert isinstance(split_table, pd.DataFrame), "split_table은 pd.DataFrame 입니다. split_rows 를 pd.DataFrame 에 넘겼는지 확인하세요."
assert list(split_table.columns) == ["질의 묶음", "모델", "질의 수", "Recall@1", "Recall@5", "MRR"], f"split_table의 열이 {list(split_table.columns)}입니다. 행 딕셔너리의 키를 질의 묶음, 모델, 질의 수, Recall@1, Recall@5, MRR 순서로 넣으세요."
assert list(split_table["질의 묶음"]) == ["학습에 나온 조문 질의", "학습에 나온 조문 질의", "처음 보는 조문 질의", "처음 보는 조문 질의"], "질의 묶음 열은 학습에 나온 조문 질의 두 행, 처음 보는 조문 질의 두 행 순서입니다."
assert list(split_table["모델"]) == ["베이스 모델", "쌍 학습 모델"] * 2, "모델 열은 묶음마다 베이스 모델, 쌍 학습 모델 순서입니다."
assert list(split_table["질의 수"]) == [37, 37, 9, 9], f"질의 수가 {list(split_table['질의 수'])}입니다. 묶음의 True 개수를 넣으세요."
for _, row in split_table.iterrows():
    expected_scores = retrieval_scores(model_ranks[row["모델"]][query_groups[row["질의 묶음"]]])
    assert row[["Recall@1", "Recall@5", "MRR"]].to_dict() == expected_scores, "각 행은 해당 모델·질의 묶음의 순위만 골라 계산해야 합니다. 묶음 점수가 같아지는 것은 허용합니다."
print("✅ 통과!")

## 7. 모델 저장과 새 질의 검색

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 학습한 모델을 검색과 과제 02에서 쓰기 위해, 폴더에 저장하고 다시 불러와 데이터에 없는 질의로 조문을 검색합니다.</p></div>

실행 후에는 상위 조문의 원문이 질문의 반품 사유·비용 조건에 맞는지 확인합니다. 유사도 1위가 그 조건에 직접 답하는 조문이라고 가정하지 않습니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) <code style="font-size:0.95em;color:#345473;">save_pretrained</code>로 <code style="font-size:0.95em;color:#345473;">commerce_tuned_model</code>을 <code style="font-size:0.95em;color:#345473;">output_dir / &quot;commerce_tuned&quot;</code>에 저장합니다.</li>
<li>(2) 그 폴더 경로로 불러와 <code style="font-size:0.95em;color:#345473;">device</code>에 올린 <code style="font-size:0.95em;color:#345473;">SentenceTransformer</code>를 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">loaded_model</code></strong> 에 담습니다.</li>
<li>(3) 셀에 적어 둔 <code style="font-size:0.95em;color:#345473;">new_query</code>와 조문 문장(<code style="font-size:0.95em;color:#345473;">commerce_corpus</code>의 <code style="font-size:0.95em;color:#345473;">text</code>)을 <code style="font-size:0.95em;color:#345473;">loaded_model</code>로 인코딩해(조문에는 <code style="font-size:0.95em;color:#345473;">PASSAGE_PREFIX</code>, 질의에는 <code style="font-size:0.95em;color:#345473;">QUERY_PREFIX</code>를 붙여) 유사도 행렬 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">search_sims</code></strong> 를 만듭니다. 모양은 (1, 39)입니다.</li>
<li>(4) <code style="font-size:0.95em;color:#345473;">search_sims</code>의 유사도가 높은 순서로 상위 5개 조문을 한 행씩 담아, 열이 <code style="font-size:0.95em;color:#345473;">&quot;순위&quot;</code>(1~5), <code style="font-size:0.95em;color:#345473;">&quot;조문&quot;</code>(셀에 적어 둔 <code style="font-size:0.95em;color:#345473;">doc_ids</code>의 doc_id), <code style="font-size:0.95em;color:#345473;">&quot;유사도&quot;</code>(<code style="font-size:0.95em;color:#345473;">search_sims</code>의 값을 반올림하지 않고 그대로), <code style="font-size:0.95em;color:#345473;">&quot;조문 내용&quot;</code>(조문 문장, 이 순서)인 다섯 행 테이블 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="font-size:0.95em;color:#345473;">search_table</code></strong> 을 만듭니다.</li>
</ul></div>

<details><summary>힌트</summary>

```text
접근방법: 교안 01 6-1 저장·불러오기 셀과 6-2 새 질의 검색 셀을 차례로 합친 모양입니다.
저장한 모델은 SentenceTransformer 에 모델 이름 대신 저장 폴더 경로를 넘겨 불러옵니다.
```

</details>

In [ ]:
# TODO (1): 모델을 output_dir / "commerce_tuned" 에 저장하세요
...

In [ ]:
# TODO (2): 모델 이름 자리에 저장 폴더 경로를 넣어 모델을 다시 불러오세요
loaded_model = ...

# 데이터에 없는 새 질의와 조문 id 목록(그대로 씁니다)
new_query = "단순 변심으로 반품하면 돌려보내는 택배비는 누가 내나요?"
doc_ids = [doc["doc_id"] for doc in commerce_corpus]

# TODO (3): 불러온 모델로 조문 벡터를 다시 만들고, 새 질의와의 유사도 (1, 39) 를 계산하세요
search_doc_emb = ...
search_sims = ...

search_rows = []
for rank, j in enumerate(np.argsort(-search_sims[0])[:5], start=1):
    # TODO (4): 순위, 조문, 유사도, 조문 내용 행 딕셔너리를 search_rows 에 더하세요
    ...
search_table = pd.DataFrame(search_rows)
display(search_table.round(3))

In [ ]:
# [자가채점] 저장 폴더, 다시 불러온 모델, 유사도 행렬, 상위 5개 조문 테이블을 확인합니다
assert (output_dir / "commerce_tuned" / "config_sentence_transformers.json").exists(), "output/commerce_tuned 에 config_sentence_transformers.json 이 없습니다. save_pretrained 에 output_dir / \"commerce_tuned\" 경로를 넘겼는지 확인하세요."
assert isinstance(loaded_model, SentenceTransformer) and loaded_model is not commerce_tuned_model, "loaded_model은 저장 폴더에서 새로 불러온 SentenceTransformer 입니다. commerce_tuned_model 을 그대로 담지 않았는지 확인하세요."
# 저장한 가중치를 그대로 불러왔다면 두 모델의 파라미터가 모두 같습니다
for loaded_param, tuned_param in zip(loaded_model.parameters(), commerce_tuned_model.parameters()):
    assert torch.allclose(loaded_param, tuned_param), "loaded_model의 가중치가 commerce_tuned_model 과 다릅니다. MODEL_ID 가 아니라 output_dir / \"commerce_tuned\" 폴더에서 불러왔는지 확인하세요."
assert isinstance(search_sims, np.ndarray) and search_sims.shape == (1, 39), f"search_sims의 모양이 {np.shape(search_sims)}입니다. 새 질의 하나와 조문 39개의 유사도 (1, 39) 인지 확인하세요."
assert isinstance(search_table, pd.DataFrame) and list(search_table.columns) == ["순위", "조문", "유사도", "조문 내용"], f"search_table의 열이 {list(search_table.columns)}입니다. 순위, 조문, 유사도, 조문 내용 순서로 키를 넣으세요."
assert list(search_table["순위"]) == [1, 2, 3, 4, 5] and list(search_table["조문"]) == [doc_ids[j] for j in np.argsort(-search_sims[0])[:5]], "순위는 1~5 이고 조문 열은 search_sims 의 유사도가 높은 순서입니다. rank 와 doc_ids[j] 를 넣었는지 확인하세요."
for doc_id, score, doc_text in zip(search_table["조문"], search_table["유사도"], search_table["조문 내용"]):
    assert abs(score - search_sims[0, doc_ids.index(doc_id)]) < 1e-6, f"{doc_id}의 유사도 열이 search_sims 의 그 조문 값과 다릅니다. search_sims[0, j] 를 넣었는지 확인하세요."
    assert doc_text == commerce_corpus[doc_ids.index(doc_id)]["text"], f"{doc_id}의 조문 내용 열이 그 조문 문장과 다릅니다. commerce_corpus[j][\"text\"] 를 넣었는지 확인하세요."
print("✅ 통과!")

## 8. 일반 분야 검색의 퇴행 비교

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">목적:</strong> 온라인쇼핑에 맞춘 학습이 다른 분야의 검색 점수를 얼마나 바꾸었는지 확인하기 위해, 베이스 모델·쌍 학습 모델을 학습에 쓰지 않은 일반 질문 83개로 비교합니다. 점수는 내려가거나 그대로이거나 올라갈 수 있습니다.</p></div>

일반 검색 문서는 총 433개입니다. 관광 43질의는 관광 388문서, 상식 30질의는 상식 30문서, 가상 도서관 10질의는 도서관 15문서 안에서만 검색합니다. 이 작은 검색 묶음의 점수를 모델의 일반능력 전체로 해석하지 않습니다.

<div class="d74-explanation" style="font-size:17px;line-height:1.75;color:#243b53;margin:10px 0 16px;padding:14px 18px;background:#f3f6fa;border-left:4px solid #54769b;border-radius:4px;"><p style="margin:0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul>
<li>(1) 제공한 반복문에서 현재 모델 <code style="font-size:0.95em;color:#345473;">model</code>, 현재 분야의 문서 <code style="font-size:0.95em;color:#345473;">domain_corpus</code>, 질의 <code style="font-size:0.95em;color:#345473;">domain_eval</code>을 <code style="font-size:0.95em;color:#345473;">gold_ranks</code>에 넘깁니다(프리픽스는 기본값). 1부터 시작하는 정수 순위를 <code style="font-size:0.95em;color:#345473;">ranks[domain_mask]</code>에 담아, 모델마다 별도의 길이 83인 배열을 <code style="font-size:0.95em;color:#345473;">general_model_ranks</code>에 저장합니다. 분야를 합쳐 검색하거나 질의 순서를 바꾸지 않습니다.</li>
</ul></div>

준비와 집계 코드는 제공합니다. `general_score_table`은 전체·분야별 `Recall@1`, `Recall@5`, `MRR`과 베이스 대비 `Recall@5 변화(pp)`를 보여 줍니다. 변화폭은 반올림 전 원순위로 계산하며, 음수는 하락·0은 동일·양수는 상승입니다. 자가채점은 모델별 순위를 다시 계산해 대조하므로 시간이 조금 걸립니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 5-3 일반 분야 검색 셀과 같습니다.
반복문이 모델과 분야별 후보를 준비해 두므로, 해당 모델의 정답 순위를 구하는 공용 함수 호출을 채웁니다.
```

</details>

In [ ]:
# [제공] 과제만 새 커널에서 실행해도 같은 일반 평가 파일을 읽습니다.
general_corpus = load_jsonl(data_dir / "본실습" / "general_corpus.jsonl")
general_eval = load_jsonl(data_dir / "본실습" / "general_eval.jsonl")
# eval 행에는 분야 열이 없으므로 정답 doc_id로 문서의 분야를 연결합니다.
general_doc_domain = {doc["doc_id"]: doc["domain"] for doc in general_corpus}
general_domains = np.array([general_doc_domain[row["doc_id"]] for row in general_eval])
general_domain_names = ["관광", "상식", "도서관"]
general_doc_counts = {domain: sum(doc["domain"] == domain for doc in general_corpus) for domain in general_domain_names}
general_data_table = pd.DataFrame([
    {"분야": domain, "검색 문서 수": general_doc_counts[domain], "평가 질의 수": int(np.sum(general_domains == domain))}
    for domain in general_domain_names
])
display(general_data_table)

In [ ]:
# 모델과 분야를 순서대로 평가합니다. 모델마다 새 순위 배열을 만듭니다.
general_models = [("베이스 모델", base_model), ("쌍 학습 모델", commerce_tuned_model)]
general_model_ranks = {}
for name, model in general_models:
    ranks = np.empty(len(general_eval), dtype=int)
    for domain in general_domain_names:
        domain_mask = general_domains == domain
        domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
        domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
        # TODO (1): 현재 모델로 같은 분야 문서에서 질의별 정답 순위를 구하세요
        ranks[domain_mask] = ...
    general_model_ranks[name] = ranks

In [ ]:
# [제공] 교안과 같은 집계입니다. 전체는 분야별 검색 순위 83개를 합쳐 계산합니다.
base_general_ranks = general_model_ranks["베이스 모델"]
general_groups = {"전체(일반 분야)": np.full(len(general_eval), True)}
general_groups.update({domain: general_domains == domain for domain in general_domain_names})
general_rows = []
for group, mask in general_groups.items():
    before_r5 = float(np.mean(base_general_ranks[mask] <= 5))
    for name, ranks in general_model_ranks.items():
        scores = retrieval_scores(ranks[mask])
        general_rows.append({
            "평가 묶음": group, "모델": name,
            "문서 수": len(general_corpus) if group == "전체(일반 분야)" else general_doc_counts[group],
            "질의 수": int(mask.sum()), **scores,
            # 표시용으로 반올림한 Recall@5끼리 빼지 않고, 원래 순위에서 변화폭을 구합니다.
            "Recall@5 변화(pp)": round(100 * (float(np.mean(ranks[mask] <= 5)) - before_r5), 2),
        })
general_score_table = pd.DataFrame(general_rows)
display(general_score_table)

In [ ]:
# [자가채점] 순위의 형태·후보 범위와 집계의 일관성을 확인합니다. 점수의 향상·하락은 채점하지 않습니다.
expected_general_models = dict([("베이스 모델", base_model), ("쌍 학습 모델", commerce_tuned_model)])
assert list(general_model_ranks) == list(expected_general_models), "general_model_ranks에 제공한 모델 이름과 순서를 유지하세요."
assert len(general_corpus) == 433 and len(general_eval) == 83, "data/본실습의 general_corpus.jsonl과 general_eval.jsonl을 읽었는지 확인하세요."
for name, ranks in general_model_ranks.items():
    assert isinstance(ranks, np.ndarray) and ranks.shape == (83,) and np.issubdtype(ranks.dtype, np.integer), f"{name}: gold_ranks의 정수 순위 83개를 원래 질의 순서에 담으세요."
    for domain in general_domain_names:
        part = ranks[general_domains == domain]
        assert part.min() >= 1 and part.max() <= general_doc_counts[domain], f"{name} {domain}: 현재 분야의 문서만 domain_corpus로 넘기고 1부터 시작하는 순위를 담으세요."
for i, name in enumerate(general_model_ranks):
    for other in list(general_model_ranks)[:i]:
        assert not np.shares_memory(general_model_ranks[name], general_model_ranks[other]), "모델마다 새 순위 배열을 만들어야 합니다. 한 배열을 여러 모델에 재사용하지 마세요."
assert isinstance(general_score_table, pd.DataFrame) and list(general_score_table.columns) == ["평가 묶음", "모델", "문서 수", "질의 수", "Recall@1", "Recall@5", "MRR", "Recall@5 변화(pp)"], "제공한 집계 셀의 general_score_table 열과 순서를 유지하세요."
assert list(zip(general_score_table["평가 묶음"], general_score_table["모델"])) == [(group, name) for group in general_groups for name in expected_general_models], "전체, 관광, 상식, 도서관 순서로 모델마다 한 행이 있어야 합니다."
for _, row in general_score_table.iterrows():
    mask = general_groups[row["평가 묶음"]]
    assert row["질의 수"] == int(mask.sum()), "행마다 해당 묶음의 질의 수를 넣어야 합니다."
    expected_docs = 433 if row["평가 묶음"] == "전체(일반 분야)" else general_doc_counts[row["평가 묶음"]]
    assert row["문서 수"] == expected_docs, "전체는 문서 합계 433개, 분야 행은 해당 분야의 후보 수입니다."
    assert all(0 <= row[metric] <= 1 for metric in ["Recall@1", "Recall@5", "MRR"]), "세 검색 지표는 0~1 범위여야 합니다."
    after_hits = np.count_nonzero(general_model_ranks[row["모델"]][mask] <= 5)
    before_hits = np.count_nonzero(general_model_ranks["베이스 모델"][mask] <= 5)
    assert abs(row["Recall@5 변화(pp)"] - round(100 * (after_hits - before_hits) / int(mask.sum()), 2)) < 1e-9, "Recall@5 변화(pp)는 반올림한 점수의 차이가 아니라 원순위의 정답 수 차이로 계산합니다."
for name in expected_general_models:
    model_rows = general_score_table[general_score_table["모델"] == name]
    whole = model_rows[model_rows["평가 묶음"] == "전체(일반 분야)"].iloc[0]
    parts = model_rows[model_rows["평가 묶음"] != "전체(일반 분야)"]
    for metric in ["Recall@1", "Recall@5", "MRR"]:
        combined = (parts[metric] * parts["질의 수"]).sum() / parts["질의 수"].sum()
        assert abs(combined - whole[metric]) < 0.002, f"{name}: 전체 {metric}는 분야별 질의 수로 가중한 점수와 맞아야 합니다."
# 모델별 실제 검색 결과와 대조합니다. 점수 방향이나 고정 점수는 요구하지 않습니다.
for name, model in expected_general_models.items():
    for domain in general_domain_names:
        domain_mask = general_domains == domain
        domain_corpus = [doc for doc in general_corpus if doc["domain"] == domain]
        domain_eval = [row for row, keep in zip(general_eval, domain_mask) if keep]
        expected_ranks = gold_ranks(model, domain_corpus, domain_eval)
        assert np.array_equal(general_model_ranks[name][domain_mask], expected_ranks), f"{name} {domain}: 이 모델의 실제 검색 순위와 다릅니다. 현재 model과 분야별 문서·질의를 사용했는지 확인하세요."
print("✅ 통과!")